# `Custom Tools and Toolkits in LangChain`

In LangChain, **Tools** and **Toolkits** are fundamental building blocks for creating AI Agents.

The simplest mental model is:

> **Tool = One capability**
> **Toolkit = A collection of related tools**
> **Agent = Uses those tools to accomplish a goal**

For example:

```text
                    AI Agent
                       │
             ┌─────────┴─────────┐
             │                   │
          Tool 1               Toolkit
        Calculator         ┌──────┼──────┐
                           │      │      │
                        Search  Insert  Update
```

---

# 1. What is a Custom Tool?

A **custom tool** is a function that you create and expose to an LLM/Agent so that it can perform an operation that isn't available through the model itself.

For example, suppose you have a company database.

You create:

```python
def get_employee(employee_id):
    # Query database
    return employee
```

By itself, this is just a Python function.

When you expose it to LangChain as a tool, the agent can decide:

> "I need employee information, so I should call `get_employee`."

That turns your function into an **AI-callable capability**.

---

# 2. Why Do We Need Custom Tools?

Built-in tools cannot know about your application's business logic.

Imagine you are building an e-commerce AI agent.

The agent needs to perform:

```text
Get product
Get order
Track order
Cancel order
Create return
Check inventory
```

These operations belong to **your application**.

Therefore, you create custom tools:

```text
get_product()
get_order()
track_order()
cancel_order()
create_return()
check_inventory()
```

Then your agent can use them.

---

# 3. Normal Function vs Custom Tool

This distinction is important.

### Normal Python function

```python
def get_weather(city):
    return f"Weather for {city}"
```

The LLM doesn't automatically know that this function exists.

### LangChain tool

```python
from langchain.tools import tool

@tool
def get_weather(city: str):
    """Get the current weather for a city."""
    return f"Weather for {city}"
```

Now LangChain can expose metadata about the function to the model.

Conceptually:

```text
Name:
get_weather

Description:
Get the current weather for a city.

Input:
city: string
```

The model can now determine whether this tool is relevant.

---

# 4. The `@tool` Decorator

One of the simplest ways to create a custom tool is:

```python
from langchain.tools import tool

@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b
```

The important part is:

```python
@tool
```

It converts the Python function into a LangChain tool.

---

# 5. Why the Docstring Is Important

Look at this:

```python
@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b
```

The docstring:

```python
"""Add two numbers."""
```

is not merely documentation for you.

It becomes part of the tool's description that helps the model understand **when to use the tool**.

For example:

```python
@tool
def get_order(order_id: int):
    """Retrieve order details using an order ID."""
    ...
```

This is much better than:

```python
@tool
def get_order(order_id: int):
    """Get data."""
    ...
```

Tool descriptions should be precise.

---

# 6. Type Hints Matter

Consider:

```python
@tool
def get_order(order_id: int):
    """Retrieve order details using an order ID."""
```

The model/tool schema can understand:

```text
order_id → integer
```

If you have:

```python
@tool
def search_products(
    category: str,
    min_price: float,
    max_price: float
):
    """Search products by category and price range."""
```

The tool has structured inputs:

```text
category  → string
min_price → number
max_price → number
```

This is one reason **type hints are important when building LangChain tools**.

---

# 7. How the Agent Uses a Custom Tool

Suppose:

```python
@tool
def get_product(product_id: str):
    """Get product information using product ID."""
    return {
        "id": product_id,
        "name": "MacBook Air",
        "price": 999
    }
```

User asks:

> What is the price of product P101?

The agent may generate a tool call conceptually like:

```json
{
  "name": "get_product",
  "arguments": {
    "product_id": "P101"
  }
}
```

The tool executes:

```python
get_product("P101")
```

Result:

```json
{
    "id": "P101",
    "name": "MacBook Air",
    "price": 999
}
```

The result goes back to the model.

The model then responds:

> The MacBook Air costs $999.

---

# 8. Custom Tool With a Database

This is where custom tools become particularly useful.

Suppose you have MongoDB:

```python
from pymongo import MongoClient

client = MongoClient(MONGO_URI)
db = client["ecommerce"]
orders = db["orders"]
```

You can create:

```python
from langchain.tools import tool

@tool
def get_order(order_id: str):
    """Retrieve an order from the database using its order ID."""

    order = orders.find_one({
        "order_id": order_id
    })

    if not order:
        return "Order not found."

    return {
        "order_id": order["order_id"],
        "status": order["status"],
        "total": order["total"]
    }
```

Now your agent has access to your MongoDB through a controlled interface.

---

# 9. Custom Tool With an API

Suppose your application has:

```text
GET /api/products/{id}
```

You could wrap the API:

```python
import requests

@tool
def get_product(product_id: str):
    """Retrieve product information from the product API."""

    response = requests.get(
        f"https://api.example.com/products/{product_id}"
    )

    response.raise_for_status()

    return response.json()
```

Now:

```text
User
 ↓
Agent
 ↓
get_product()
 ↓
REST API
 ↓
Product data
 ↓
Agent
 ↓
Answer
```

---

# 10. Custom Tool for Business Logic

This is one of the most important real-world applications.

Suppose your company has a discount rule:

```text
If order > ₹10,000
AND customer is premium
→ 20% discount
```

You can create:

```python
@tool
def calculate_discount(
    order_amount: float,
    is_premium: bool
):
    """Calculate the applicable customer discount."""

    if order_amount > 10000 and is_premium:
        return order_amount * 0.20

    return 0
```

The LLM doesn't need to know or reproduce your business rules.

The **tool contains deterministic business logic**.

This is a very good design principle:

> Let the LLM decide **what capability is needed**, but let deterministic code perform **critical business logic**.

---

# 11. Multiple Custom Tools

A real agent usually needs multiple tools.

For example, an e-commerce agent:

```python
@tool
def get_product(product_id: str):
    """Get product information."""
    ...


@tool
def check_inventory(product_id: str):
    """Check product inventory."""
    ...


@tool
def get_order(order_id: str):
    """Get order information."""
    ...


@tool
def track_order(order_id: str):
    """Track an order."""
    ...
```

You can collect them:

```python
tools = [
    get_product,
    check_inventory,
    get_order,
    track_order
]
```

Then provide these tools to your agent/model.

---

# 12. What Is a Toolkit?

Now we move to the second concept.

A **Toolkit is a collection of related tools designed to work together for a particular domain or system.**

For example:

```text
SQL Toolkit
│
├── List tables
├── Get schema
├── Execute query
└── Query database
```

Or:

```text
GitHub Toolkit
│
├── Search repositories
├── Get issues
├── Create issue
└── Manage pull requests
```

Or:

```text
File Toolkit
│
├── Read file
├── Write file
├── Search file
└── Delete file
```

The exact available toolkit classes and APIs depend on the LangChain/LangGraph version and integration package you're using, but the conceptual distinction remains the same.

---

# 13. Tool vs Toolkit

This is an important interview question.

### Tool

One capability.

```text
get_order()
```

### Toolkit

Multiple related capabilities.

```text
Order Toolkit
│
├── get_order()
├── cancel_order()
├── track_order()
└── create_return()
```

So:

> **Tool = individual function/capability**
> **Toolkit = group of related tools**

---

# 14. Why Use a Toolkit?

Imagine creating an SQL agent.

You could manually create:

```python
list_tables()
get_schema()
execute_query()
```

and pass all of them individually.

But a database integration may already provide a toolkit containing the tools required for interacting with that database.

Conceptually:

```text
Database Toolkit
        │
        ├── list_tables
        ├── schema
        ├── query
        └── inspect
```

This reduces boilerplate and gives you a standardized collection of capabilities.

---

# 15. Example: SQL Toolkit

A typical SQL agent architecture looks like:

```text
                    SQL Agent
                       │
                       ▼
                 SQL Toolkit
                       │
          ┌────────────┼────────────┐
          ▼            ▼            ▼
     List Tables   Get Schema   Execute Query
          │            │            │
          └────────────┼────────────┘
                       ▼
                    SQL DB
```

User:

> How many customers placed orders this month?

The agent may:

```text
1. Inspect database
2. Identify relevant tables
3. Inspect schema
4. Generate SQL
5. Execute SQL
6. Interpret result
7. Answer user
```

The toolkit supplies the database capabilities.

---

# 16. Toolkit Does Not Mean Agent

This distinction is critical.

A toolkit itself does **not necessarily reason**.

For example:

```text
SQL Toolkit
    ↓
Provides tools
```

The agent:

```text
Agent
    ↓
Decides which SQL tool to call
```

So:

```text
Toolkit → collection of capabilities

Agent → decision-maker

LLM → reasoning/generation engine
```

Together:

```text
LLM
 ↓
Agent
 ↓
Toolkit
 ↓
Tools
 ↓
External System
```

---

# 17. Custom Toolkit

You can also create your own toolkit.

Suppose you're building an HR Agent.

You create:

```text
HR Toolkit
│
├── get_employee()
├── get_salary()
├── get_leave_balance()
├── apply_leave()
└── get_department()
```

Conceptually:

```python
class HRToolkit:

    def get_tools(self):
        return [
            get_employee,
            get_salary,
            get_leave_balance,
            apply_leave,
            get_department
        ]
```

Then:

```python
hr_toolkit = HRToolkit()

tools = hr_toolkit.get_tools()
```

The exact implementation can vary, but this is the core idea.

---

# 18. Custom Tools vs Custom Toolkit

Suppose you're building an **E-commerce Agent**.

### Custom tools

```text
get_product()
get_order()
track_order()
cancel_order()
```

### Custom toolkit

```text
EcommerceToolkit
│
├── get_product()
├── get_order()
├── track_order()
└── cancel_order()
```

If you only need one capability:

```text
Custom Tool
```

If you have a domain containing many related capabilities:

```text
Custom Toolkit
```

---

# 19. Toolkit Architecture

A useful architecture is:

```text
                    AI Agent
                       │
                       ▼
                  Tool Registry
                       │
          ┌────────────┼─────────────┐
          │            │             │
          ▼            ▼             ▼
     HR Toolkit    SQL Toolkit   Web Toolkit
          │            │             │
       ┌──┼──┐       ┌─┼──┐        ┌─┼──┐
       │  │  │       │ │  │        │ │  │
      Get Leave Salary Query Schema Search Fetch
```

An agent can potentially have multiple toolkits.

---

# 20. Toolkits in a Production System

Imagine a customer-service agent:

```text
Customer Support Agent
│
├── CRM Toolkit
│   ├── get_customer()
│   ├── update_customer()
│   └── get_customer_history()
│
├── Order Toolkit
│   ├── get_order()
│   ├── track_order()
│   └── cancel_order()
│
├── Knowledge Toolkit
│   ├── search_docs()
│   └── get_policy()
│
└── Communication Toolkit
    ├── send_email()
    └── send_sms()
```

Now the agent has a much richer action space.

---

# 21. Important: Don't Give an Agent Every Tool

More tools do not automatically make an agent better.

Suppose you give it:

```text
50 tools
```

Some tools may overlap.

The model may struggle to choose correctly.

For example:

```text
search_customer()
find_customer()
lookup_customer()
get_customer()
query_customer()
```

These are too similar.

A better design is:

```text
get_customer()
search_customers()
```

with clear descriptions.

---

# 22. Tool Granularity

Tool design has an important tradeoff.

### Too large

```text
do_everything()
```

Problem:

* difficult to understand
* difficult to validate
* difficult to secure
* difficult to test

### Too small

```text
open_connection()
authenticate()
create_cursor()
execute()
close_cursor()
```

Now the agent has to manage low-level implementation details.

### Better

```text
execute_database_query()
```

The tool encapsulates implementation complexity.

So a good principle is:

> **Expose meaningful business capabilities, not unnecessary implementation details.**

---

# 23. Tool Description Example

Bad:

```python
@tool
def process(data):
    """Process data."""
```

Better:

```python
@tool
def get_order_status(order_id: str):
    """
    Retrieve the current shipping status of an order.

    Use this when the user asks where an order is
    or whether an order has shipped.
    """
```

This gives the model stronger guidance.

---

# 24. Tool Validation

Custom tools should validate input.

For example:

```python
@tool
def cancel_order(order_id: str):
    """Cancel an order if it is eligible for cancellation."""

    if not order_id:
        return "Order ID is required."

    order = get_order_from_db(order_id)

    if not order:
        return "Order not found."

    if order["status"] == "delivered":
        return "Delivered orders cannot be cancelled."

    ...
```

The agent should not be trusted to enforce business rules purely through natural-language reasoning.

---

# 25. Tools and Security

This becomes extremely important in production.

Consider:

```python
@tool
def delete_user(user_id: str):
    ...
```

Giving this capability to an LLM is potentially dangerous.

You should consider:

```text
Authentication
Authorization
Input validation
Rate limiting
Audit logging
Confirmation
Human approval
Idempotency
```

Especially for tools that:

```text
DELETE
UPDATE
PAY
SEND
DEPLOY
```

---

# 26. Read vs Write Tools

A useful classification:

### Read tools

```text
get_user()
get_order()
search_docs()
get_inventory()
search_products()
```

Generally lower risk.

### Write tools

```text
create_order()
update_user()
cancel_order()
send_email()
make_payment()
```

Higher risk.

### Destructive tools

```text
delete_user()
delete_database()
delete_file()
```

Highest risk.

The more powerful the tool, the stronger the guardrails should be.

---

# 27. Custom Tool + Agent Example

Conceptually:

```python
from langchain.tools import tool

@tool
def get_order(order_id: str):
    """Get order information using an order ID."""

    return {
        "order_id": order_id,
        "status": "shipped",
        "tracking_id": "TRK123"
    }


@tool
def track_order(tracking_id: str):
    """Get the latest shipping location of a package."""

    return {
        "tracking_id": tracking_id,
        "location": "Delhi",
        "status": "In Transit"
    }
```

Then:

```python
tools = [
    get_order,
    track_order
]
```

The agent has two capabilities.

---

# 28. Multi-Step Tool Usage

User:

> Where is order 123?

The agent may do:

```text
get_order("123")
        ↓
tracking_id = "TRK123"
        ↓
track_order("TRK123")
        ↓
Result
```

This is an important agent pattern:

> **The output of one tool can become the input to another tool.**

This enables multi-step workflows.

---

# 29. Tool Chaining

For example:

```text
User
 ↓
get_order()
 ↓
tracking_id
 ↓
track_order()
 ↓
location
 ↓
get_delivery_estimate()
 ↓
delivery date
 ↓
Final Answer
```

Notice that the agent is coordinating multiple capabilities.

This is where agents become much more powerful than simple function calls.

---

# 30. Custom Tool Development Workflow

When you build a custom tool, follow this process:

```text
1. Identify capability
        ↓
2. Create Python function
        ↓
3. Add type hints
        ↓
4. Add clear description
        ↓
5. Add validation
        ↓
6. Convert to LangChain tool
        ↓
7. Test tool independently
        ↓
8. Give tool to LLM
        ↓
9. Test tool selection
        ↓
10. Add guardrails
```

---

# 31. Custom Toolkit Development Workflow

For a toolkit:

```text
Identify domain
      ↓
Identify capabilities
      ↓
Create individual tools
      ↓
Validate each tool
      ↓
Group related tools
      ↓
Create toolkit
      ↓
Expose get_tools()
      ↓
Give tools to Agent
      ↓
Test complete workflow
```

---

# 32. Real Project Example

For your GenAI learning, a strong project would be an:

## E-commerce Shopping Agent

Create these custom tools:

```text
Product Tools
├── search_products()
├── get_product_details()
└── check_inventory()

Order Tools
├── get_order()
├── track_order()
└── cancel_order()

Customer Tools
├── get_customer()
└── get_order_history()

Recommendation Tools
└── recommend_products()
```

Then group them:

```text
Ecommerce Toolkit
│
├── Product Tools
├── Order Tools
├── Customer Tools
└── Recommendation Tools
```

The agent can answer:

> Find me a laptop under ₹80,000.

```text
search_products()
```

Then:

> Is the Lenovo model in stock?

```text
check_inventory()
```

Then:

> Where is my previous order?

```text
get_order()
track_order()
```

Then:

> Cancel it.

```text
cancel_order()
```

That gives you a very realistic **tool-using AI agent project**.

---

# 33. Tool vs Toolkit vs Agent vs LLM

Keep this table in your notes:

| Concept             | Responsibility                   | Example                |
| ------------------- | -------------------------------- | ---------------------- |
| **LLM**             | Reasoning / generation           | GPT model              |
| **Tool**            | One capability                   | `get_order()`          |
| **Toolkit**         | Group of related tools           | `OrderToolkit`         |
| **Agent**           | Decides what actions to take     | Customer Support Agent |
| **External System** | Performs/stores actual operation | MongoDB/API            |
| **Memory**          | Maintains relevant state/history | Conversation history   |
| **RAG**             | Retrieves relevant knowledge     | Vector DB              |

---

# 34. Complete Mental Model

```text
                         USER
                           │
                           ▼
                    ┌─────────────┐
                    │     LLM     │
                    └──────┬──────┘
                           │
                           ▼
                    ┌─────────────┐
                    │    AGENT    │
                    │             │
                    │ Tool Select │
                    └──────┬──────┘
                           │
             ┌─────────────┼──────────────┐
             │             │              │
             ▼             ▼              ▼
       Custom Tool    Toolkit Tool    RAG Tool
             │             │              │
             │       ┌─────┼─────┐        │
             │       │     │     │        │
             │      Tool  Tool  Tool      │
             │       │     │     │        │
             ▼       ▼     ▼     ▼        ▼
          REST API  DB   API   Service  Vector DB
             │       │     │     │        │
             └───────┴─────┴─────┴────────┘
                           │
                           ▼
                      Tool Result
                           │
                           ▼
                          LLM
                           │
                           ▼
                     Final Response
```

## The core idea

**Custom Tool:**

> "I have created a specific capability for my agent."

**Toolkit:**

> "I have grouped multiple related capabilities into a reusable collection."

**Agent:**

> "I decide which capability to use and in what sequence."

**LLM:**

> "I understand the user's goal, reason about available actions, generate tool calls, and interpret the results."

For LangChain specifically, the next concept to learn after this is **Tool Calling → `bind_tools()` → Agent creation → Agent loop → multiple custom tools → tool error handling → human approval/guardrails**.
